In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, confusion_matrix,  accuracy_score, precision_recall_fscore_support,r2_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import shap
import os
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
import itertools
import torch.optim as optim
import seaborn as sns

In [ ]:
#Bringing all the data into the dataframe

DATA_DIR = "unzipped_csvs/cleaned_dataset/data"
METADATA_PATH = "metadata.csv"

try:
    metadata = pd.read_csv(METADATA_PATH)
    print(f"Metadata loaded from {METADATA_PATH}")
except Exception as e:
    print(f"Error loading metadata: {e}")

all_data = []

for idx, row in metadata.iterrows():
    filename = row["filename"]
    filepath = os.path.join(DATA_DIR, filename)
    print(f"Opening {filename}...")

    try:
        df = pd.read_csv(filepath)
        for col in metadata.columns:
            df[col] = row[col]
        all_data.append(df)
    except Exception as e:
        print(f"Error reading {filename}: {e}")

if all_data:
    full_df = pd.concat(all_data, ignore_index=True)
else:
    print("No data was loaded.")


In [ ]:
#Uses Regex to extract 00001 to 1, etc, capturing the digits, and drop leading zeros for cycle index
full_df["cycle_index"] = (full_df.filename.str.extract(r"^0*(\d+)\.csv$", expand=False).astype(int))
full_df.head()

In [ ]:
#Creates a local cycle counter for each battery
full_df["cycle"] = (full_df["cycle_index"] - full_df.groupby("battery_id")["cycle_index"].transform("min") + 1)

#Check the new cycle numbers
unique = (full_df[["battery_id","cycle"]].drop_duplicates().groupby("battery_id")["cycle"].agg(["min","max","count"]).sort_index())
print(unique)

In [ ]:
MIN_VALID_Q=0.5
N_PROBE_CYCLES=5
battery_caps={}
for bid,df_batt in full_df.groupby("battery_id"):
    q_list=[]
    for cyc in range(1,N_PROBE_CYCLES+1):
        d=df_batt[(df_batt["cycle"]==cyc)&(df_batt["Current_measured"]<0)].sort_values("Time")
        if d.empty: continue
        cut=d[d["Voltage_measured"]<2.7].index
        if len(cut): d=d.loc[:cut.min()]
        dt_hr=d["Time"].diff().fillna(0)/3600.0
        q_ah=-(d["Current_measured"]*dt_hr).sum()
        if q_ah>=MIN_VALID_Q: q_list.append(q_ah)
    if q_list: battery_caps[bid]=max(q_list)

records=[]
for (bid,cyc),df_c in full_df.groupby(["battery_id","cycle"]):
    q_nom=battery_caps.get(bid)
    if q_nom is None: continue
    d=df_c[df_c["Current_measured"]<0].sort_values("Time")
    
    if d.empty: continue
    cut=d[d["Voltage_measured"]<2.7].index
    
    if len(cut): d=d.loc[:cut.min()]
    dt_hr=d["Time"].diff().fillna(0)/3600.0
    d["dQ"]=-(d["Current_measured"]*dt_hr)
    d["Qcum"]=d["dQ"].cumsum()
    q_cycle=d["dQ"].sum()
    
    if q_cycle<MIN_VALID_Q: continue
    soh=100*q_cycle/q_nom
    d["SoC"]=100*d["Qcum"]/q_cycle
    soc_grid=np.linspace(0,100,20)
    interp=lambda col: np.interp(soc_grid,d["SoC"],d[col])
    amb_temp=df_c["ambient_temperature"].iloc[0]
    df_out=pd.DataFrame({"SoC":soc_grid.clip(0,100), "Voltage_measured":interp("Voltage_measured"),
        "Current_measured":interp("Current_measured"), "Temperature_measured":interp("Temperature_measured"),
        "ambient_temperature":amb_temp, "SoH":soh})
    df_out["battery_id"]=bid
    df_out["cycle"]=cyc
    records.append(df_out)

battery_health=pd.concat(records,ignore_index=True)
plt.figure(figsize=(5,3))
sns.histplot(battery_health["SoH"],bins=30,color="C2",edgecolor="black")
plt.title("Cleaned SoH Distribution")
plt.xlabel("SoH (%)")
plt.show()

In [ ]:
battery_health = battery_health[battery_health["SoH"] <= 100.0]

plt.figure(figsize=(5,3))
sns.histplot(battery_health["SoH"], bins=30, color="C2")
plt.title("SoH Distribution")
plt.xlabel("SoH (%)"); plt.show()

In [ ]:
plt.figure(figsize=(5,3))
sns.histplot(battery_health["SoC"], bins=30, color="C2")
plt.title("SoC Distribution")
plt.xlabel("SoC (%)"); plt.show()

In [ ]:
#Train CNNRegressor with Battery‐Grouped Split 

feature_cols = ['Voltage_measured','Current_measured','Temperature_measured','SoC','ambient_temperature']
X_list, y_list, batt_ids = [], [], []

for (bid, cyc), grp in battery_health.groupby(['battery_id','cycle']):
    X_list.append(grp[feature_cols].values.T.astype(np.float32))
    y_list.append(grp['SoH'].iloc[0] / 100.0)
    batt_ids.append(bid)

X = np.stack(X_list)                     
y = np.array(y_list, dtype=np.float32)   
batt_ids = np.array(batt_ids)

#Group‐wise split for no battery leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=batt_ids))
X_tr, X_val = X[train_idx], X[val_idx]
y_tr, y_val = y[train_idx], y[val_idx]

#normalize train data
means = X_tr.mean(axis=(0,2), keepdims=True)
stds = X_tr.std(axis=(0,2), keepdims=True)
X_tr = (X_tr - means) / (stds + 1e-6)
X_val = (X_val - means) / (stds + 1e-6)

#Dataloaders
train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

class CNNRegressor(nn.Module):
    def __init__(self, in_ch=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(64,128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(128,256,kernel_size=3,padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(256,1)

    def forward(self,x):
        x = self.features(x).squeeze(-1)
        return self.head(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNRegressor().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
criterion = nn.L1Loss() 

train_maes, val_maes = [], []
for epoch in range(60):
    model.train()
    train_acc = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_acc += loss.item() * xb.size(0)
    train_mae = train_acc / len(train_loader.dataset)
    train_maes.append(train_mae)

    model.eval()
    val_acc = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_acc += criterion(model(xb), yb).item() * xb.size(0)
    val_mae = val_acc / len(val_loader.dataset)
    val_maes.append(val_mae)

    scheduler.step(val_mae)
    if epoch==1 or epoch%10==0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:02d} | Train MAE: {train_mae*100:.2f}% | Val MAE: {val_mae*100:.2f}% | LR: {lr:.1e}")

plt.plot(np.array(train_maes)*100, label="Train")
plt.plot(np.array(val_maes)*100, label="Val")
plt.xlabel("Epoch"); plt.ylabel("MAE (%)"); plt.legend(); plt.show()

model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_val).to(device)).cpu().numpy()*100
y_true = y_val*100
print("Final Val MAE: %.2f%%" % (np.mean(np.abs(y_pred - y_true))))


In [ ]:
#Plotting Predicted vs Actual SoH 

#Recover SoH in %
y_true = y_val * 100
mae = np.mean(np.abs(y_pred - y_true))
print(f"Scatter MAE: {mae:.2f}%")

plt.figure(figsize=(6,6))
plt.scatter(y_true, y_pred, alpha=0.6)
lims = [max(60, min(y_true.min(), y_pred.min())),min(100, max(y_true.max(), y_pred.max()))]
plt.plot(lims, lims, 'k--', linewidth=1)
plt.xlabel("True SoH (%)")
plt.ylabel("Predicted SoH (%)")
plt.title("True vs Predicted SoH")
plt.xlim(60, 100)
plt.ylim(60, 100)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
#Grid Search Parameter Ranges
lrs = [1e-3, 5e-4, 1e-4]
wds = [1e-4, 1e-5]
drops = [0.1, 0.3]
grid = list(itertools.product(lrs, wds, drops))

class CNNRegSearch(nn.Module):
    def __init__(self, in_ch=5, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch,64,3,padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64,128,3,padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
            nn.Flatten(), nn.Linear(128,1) 
        )
    def forward(self,x):
        return self.net(x).squeeze(-1)

results = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.L1Loss() #MAE

for lr, wd, dr in grid:
    model = CNNRegSearch(dropout=dr).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for epoch in range(20):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds.append(model(xb).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    mae = mean_absolute_error(trues, preds)

    results.append({'lr': lr,'weight_decay': wd,'dropout': dr,'val_mae': mae})

df = pd.DataFrame(results)
display(df)

#Heatmaps
for wd in sorted(df.weight_decay.unique()):
    sub = df[df.weight_decay == wd]
    pivot = sub.pivot(index='dropout', columns='lr', values='val_mae')
    plt.figure()
    plt.imshow(pivot, aspect='auto')
    plt.title(f"Val MAE (weight_decay={wd})")
    plt.xlabel("Learning Rate")
    plt.ylabel("Dropout Rate")
    plt.xticks(np.arange(len(pivot.columns)), pivot.columns)
    plt.yticks(np.arange(len(pivot.index)), pivot.index)
    plt.colorbar(label="MAE")
    plt.show()

In [ ]:
#Architecture & Scheduler Grid Search
depths = [2, 3]
widths = [64, 128]
kernels = [3, 5]
param_grid = list(itertools.product(depths, widths, kernels))

class CNNParam(nn.Module):
    def __init__(self, in_ch=5, depth=2, width=64, kernel=3, dropout=0.2):
        super().__init__()
        layers = []
        ch = in_ch
        for _ in range(depth):
            layers += [
                nn.Conv1d(ch, width, kernel_size=kernel, padding=kernel//2),
                nn.BatchNorm1d(width),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            ch = width
        layers += [nn.AdaptiveAvgPool1d(1), nn.Flatten()]
        self.features = nn.Sequential(*layers)
        self.regressor = nn.Sequential(nn.Linear(width, 1))
    def forward(self, x):
        x = self.features(x)
        return self.regressor(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.L1Loss()  
num_epochs = 20
results = []

for depth, width, kernel in param_grid:
    model = CNNParam(depth=depth, width=width, kernel=kernel).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer,max_lr=5e-4,steps_per_epoch=len(train_loader),epochs=num_epochs)

    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
    model.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_preds.append(model(xb).cpu().numpy())
            val_trues.append(yb.cpu().numpy())
    val_preds = np.concatenate(val_preds)
    val_trues = np.concatenate(val_trues)
    mae = mean_absolute_error(val_trues, val_preds)
    results.append({'depth': depth,'width': width,'kernel': kernel,'val_mae': mae})

df_results = pd.DataFrame(results)
print("\nArchitecture + Scheduler Grid Search Results:\n")
display(df_results)

#Heatmaps
for depth in depths:
    sub = df_results[df_results['depth'] == depth]
    pivot = sub.pivot(index='width', columns='kernel', values='val_mae')
    plt.figure(figsize=(5,4))
    plt.imshow(pivot, aspect='auto')
    plt.title(f"Val MAE (depth={depth})")
    plt.xlabel("Kernel Size")
    plt.ylabel("Channel Width")
    plt.xticks(np.arange(len(pivot.columns)), pivot.columns)
    plt.yticks(np.arange(len(pivot.index)), pivot.index)
    plt.colorbar(label="MAE")
    plt.show()